# Tables 2–4 & Figures 7–9 (data preparation): Conformal inference

This notebook **computes** the treatment-effect estimates and uncertainty intervals used in:

- **Tables 2–4**: treatment effect estimates and 90% intervals
- **Figures 7–9**: confidence bands (plotted in the next notebook)

It runs three empirical datasets (Germany / Basque / California) and evaluates:

- MMSCM for \(G \in \{2,3,5,10,50,100\}\)
- Abadie
- DiSCo

The outputs are saved as NumPy files under `result/`:

- `result/germany_res.npy`
- `result/basque_res.npy`
- `result/tabacco_res.npy`

## How to run

1. Make sure the `result/` directory exists (this notebook creates it automatically).
2. Run all cells from top to bottom.
3. Then open the companion notebook:
   **`Tables2_3_4_Fig07_09_Render_Tables_and_Confidence_Bands.ipynb`**.

## Notes

- The numerical logic and plotting parameters are preserved from the original code.
- Only minimal cleanup (directory creation, removing unused imports, adding this header) was applied.


In [1]:
import numpy as np
import pandas as pd
import scipy
import matplotlib.pyplot as plt

from mmscm import *

import os
os.makedirs("result", exist_ok=True)

DATA_DIR = "dataset/"

In [1]:
data = pd.read_csv(DATA_DIR + "german_reunification.csv")
data = data.drop(columns="code", axis=1)

target_unit_var = "country"
target_unit = "West Germany"
target_year_var = "year"
target_year = 1990

target_outcome_var = "gdp"

NameError: name 'pd' is not defined

In [2]:
split_num  = 100

mmscm_res_list = []


for poly in [2, 3, 5, 10, 50, 100]:
    mmscm_res = MMSCM(data, "MMSCM", target_unit_var, target_unit, target_outcome_var, target_year_var, target_year, poly=poly)
    mmscm_res.train_param()
    treated_unit_outcome, scm_pred = mmscm_res.predict(bias=False)

    treatment_effect_list = []


    count = 0

    for effect_year in range(1, (mmscm_res.year[-1] - target_year)+1):
        treatment_effect_temp, target_treated_unit_outcome_temp, target_counterfactual_outcome_temp = mmscm_res.treatment_effect(effect_year)
        treatment_effect_list.append(treatment_effect_temp)
        count += 1

    treatment_effect_list = np.array(treatment_effect_list).T[0]
    treatment_effect_abs = np.max(np.abs(treatment_effect_list))
    treatment_effect = np.mean(treatment_effect_list)
    range_interval_list = []

    for i in range(len(treatment_effect_list)):
        treatment_effect_temp = treatment_effect_list[i]
        treatment_effect_min = treatment_effect_temp - 2*treatment_effect_temp
        treatment_effect_max = treatment_effect_temp + 2*treatment_effect_temp
        range_interval_list_temp = np.linspace(treatment_effect_min, treatment_effect_max, split_num)
        range_interval_list.append(range_interval_list_temp)

    range_interval_list = np.array(range_interval_list).T

    #conformal_base_value = np.abs(np.array(treatment_effect_list).T[0]).sum() / np.sqrt(len(treatment_effect_list))

    interval = mmscm_res.conformal_inference(range_interval_list)

    if len(interval) == 0:
        interval = np.append(interval, treatment_effect_list)

    mmscm_res = np.array([treatment_effect_list, np.min(interval, axis=0), np.max(interval, axis=0)])

    mmscm_res_list.append(mmscm_res)

#---------------------------------------------------------------------------------------------------------------


mmscm_res = MMSCM(data, "Abadie", target_unit_var, target_unit, target_outcome_var, target_year_var, target_year)
mmscm_res.train_param()
treated_unit_outcome, scm_pred = mmscm_res.predict(bias=False)

treatment_effect_list = []


count = 0

for effect_year in range(1, (mmscm_res.year[-1] - target_year)+1):
    treatment_effect_temp, target_treated_unit_outcome_temp, target_counterfactual_outcome_temp = mmscm_res.treatment_effect(effect_year)
    treatment_effect_list.append(treatment_effect_temp)
    count += 1

treatment_effect_list = np.array(treatment_effect_list).T[0]
treatment_effect_abs = np.max(np.abs(treatment_effect_list))
treatment_effect = np.mean(treatment_effect_list)
range_interval_list = []

for i in range(len(treatment_effect_list)):
    treatment_effect_temp = treatment_effect_list[i]
    treatment_effect_min = treatment_effect_temp - 2*treatment_effect_temp
    treatment_effect_max = treatment_effect_temp + 2*treatment_effect_temp
    range_interval_list_temp = np.linspace(treatment_effect_min, treatment_effect_max, split_num)
    range_interval_list.append(range_interval_list_temp)
    
range_interval_list = np.array(range_interval_list).T

#conformal_base_value = np.abs(np.array(treatment_effect_list).T[0]).sum() / np.sqrt(len(treatment_effect_list))

interval = mmscm_res.conformal_inference(range_interval_list)

if len(interval) == 0:
    interval = np.append(interval, treatment_effect_list)

mmscm_res = np.array([treatment_effect_list, np.min(interval, axis=0), np.max(interval, axis=0)])

mmscm_res_list.append(mmscm_res)


#---------------------------------------------------------------------------------------------------------------


mmscm_res = MMSCM(data, "DiSCo", target_unit_var, target_unit, target_outcome_var, target_year_var, target_year)
mmscm_res.train_param()
treated_unit_outcome, scm_pred = mmscm_res.predict(bias=False)

treatment_effect_list = []


count = 0

for effect_year in range(1, (mmscm_res.year[-1] - target_year)+1):
    treatment_effect_temp, target_treated_unit_outcome_temp, target_counterfactual_outcome_temp = mmscm_res.treatment_effect(effect_year)
    treatment_effect_list.append(treatment_effect_temp)
    count += 1

treatment_effect_list = np.array(treatment_effect_list).T[0]
treatment_effect_abs = np.max(np.abs(treatment_effect_list))
treatment_effect = np.mean(treatment_effect_list)
range_interval_list = []

for i in range(len(treatment_effect_list)):
    treatment_effect_temp = treatment_effect_list[i]
    treatment_effect_min = treatment_effect_temp - 2*treatment_effect_temp
    treatment_effect_max = treatment_effect_temp + 2*treatment_effect_temp
    range_interval_list_temp = np.linspace(treatment_effect_min, treatment_effect_max, split_num)
    range_interval_list.append(range_interval_list_temp)
    
range_interval_list = np.array(range_interval_list).T

#conformal_base_value = np.abs(np.array(treatment_effect_list).T[0]).sum() / np.sqrt(len(treatment_effect_list))

interval = mmscm_res.conformal_inference(range_interval_list)

if len(interval) == 0:
    interval = np.append(interval, treatment_effect_list)

mmscm_res = np.array([treatment_effect_list, np.min(interval, axis=0), np.max(interval, axis=0)])

mmscm_res_list.append(mmscm_res)







mmscm_res_list

np.save("result/germany_res.npy", mmscm_res_list)

NameError: name 'MMSCM' is not defined

In [3]:
###############################################################################
# SCPI Python Package
# Script for Empirical Illustration - Single Treated Unit
# Authors: Matias D. Cattaneo, Yingjie Feng, Filippo Palomba and Rocio Titiunik
###############################################################################

########################################
# Load SCPI_PKG package
import pandas
import numpy
import os
from copy import deepcopy
from warnings import filterwarnings
from scpi_pkg.scdata import scdata
from scpi_pkg.scdataMulti import scdataMulti
from scpi_pkg.scest import scest
from scpi_pkg.scpi import scpi
from scpi_pkg.scplot import scplot
from scpi_pkg.scplotMulti import scplotMulti

########################################
# Load database

filterwarnings("ignore")

##############################################################################
# SINGLE TREATED UNIT
##############################################################################

########################################
# Set options for data preparation
id_var = 'country'
outcome_var = 'gdp'
time_var = 'year'
features = None
cov_adj = None
period_pre = numpy.arange(1960, 1991)
period_post = numpy.arange(1991, 2004)
unit_tr = 'West Germany'
unit_co = list(set(data[id_var].to_list()))
unit_co = [cou for cou in unit_co if cou != 'West Germany']
constant = False
cointegrated_data = True

data_prep = scdata(df=data, id_var=id_var, time_var=time_var,
                   outcome_var=outcome_var, period_pre=period_pre,
                   period_post=period_post, unit_tr=unit_tr,
                   unit_co=unit_co, features=features, cov_adj=cov_adj,
                   cointegrated_data=cointegrated_data, constant=constant)


####################################
# SC - point estimation with simplex
est_si = scest(data_prep, w_constr={'name': "simplex"})
print(est_si)

est_si2 = scest(data_prep, w_constr={'p': 'L1', 'dir': '==', 'Q': 1, 'lb': 0})
print(est_si2)


####################################
# Set options for inference
w_constr = {'name': 'simplex', 'Q': 1}
u_missp = True
u_sigma = "HC1"
u_order = 1
u_lags = 0
e_method = "gaussian"
e_order = 1
e_lags = 0
e_alpha = 0.05
u_alpha = 0.05
sims = 200
cores = 1

numpy.random.seed(8894)
pi_si = scpi(data_prep, sims=sims, w_constr=w_constr, u_order=u_order, u_lags=u_lags,
             e_order=e_order, e_lags=e_lags, e_method=e_method, u_missp=u_missp,
             u_sigma=u_sigma, cores=cores, e_alpha=e_alpha, u_alpha=u_alpha)
print(pi_si)

mmscm_res = np.array([pi_si.Y_post["West Germany"].values - pi_si.Y_post_fit["A"].values, pi_si.Y_post["West Germany"].values - pi_si.CI_all_gaussian["Lower"].values, pi_si.Y_post["West Germany"].values - pi_si.CI_all_gaussian["Upper"].values])
mmscm_res_list.append(mmscm_res)
np.save("result/germany_res.npy", mmscm_res_list)

NameError: name 'data' is not defined

In [ ]:
#Import data
data = pd.read_csv(DATA_DIR + "basque_data.csv")

target_unit_var = "regionname"
target_unit = "Basque Country (Pais Vasco)"
target_year_var = "year"
target_year = 1970

target_outcome_var = "gdpcap"

In [ ]:
split_num  = 100

mmscm_res_list = []


for poly in [2, 3, 5, 10, 50, 100]:
    mmscm_res = MMSCM(data, "MMSCM", target_unit_var, target_unit, target_outcome_var, target_year_var, target_year, poly=poly)
    mmscm_res.train_param()
    treated_unit_outcome, scm_pred = mmscm_res.predict(bias=False)

    treatment_effect_list = []


    count = 0

    for effect_year in range(1, (mmscm_res.year[-1] - target_year)+1):
        treatment_effect_temp, target_treated_unit_outcome_temp, target_counterfactual_outcome_temp = mmscm_res.treatment_effect(effect_year)
        treatment_effect_list.append(treatment_effect_temp)
        count += 1

    treatment_effect_list = np.array(treatment_effect_list).T[0]
    treatment_effect_abs = np.max(np.abs(treatment_effect_list))
    treatment_effect = np.mean(treatment_effect_list)
    range_interval_list = []

    for i in range(len(treatment_effect_list)):
        treatment_effect_temp = treatment_effect_list[i]
        treatment_effect_min = treatment_effect_temp - 2*treatment_effect_temp
        treatment_effect_max = treatment_effect_temp + 2*treatment_effect_temp
        range_interval_list_temp = np.linspace(treatment_effect_min, treatment_effect_max, split_num)
        range_interval_list.append(range_interval_list_temp)

    range_interval_list = np.array(range_interval_list).T

    #conformal_base_value = np.abs(np.array(treatment_effect_list).T[0]).sum() / np.sqrt(len(treatment_effect_list))

    interval = mmscm_res.conformal_inference(range_interval_list)

    if len(interval) == 0:
        interval = np.append(interval, treatment_effect_list)

    mmscm_res = np.array([treatment_effect_list, np.min(interval, axis=0), np.max(interval, axis=0)])

    mmscm_res_list.append(mmscm_res)

#---------------------------------------------------------------------------------------------------------------


mmscm_res = MMSCM(data, "Abadie", target_unit_var, target_unit, target_outcome_var, target_year_var, target_year)
mmscm_res.train_param()
treated_unit_outcome, scm_pred = mmscm_res.predict(bias=False)

treatment_effect_list = []


count = 0

for effect_year in range(1, (mmscm_res.year[-1] - target_year)+1):
    treatment_effect_temp, target_treated_unit_outcome_temp, target_counterfactual_outcome_temp = mmscm_res.treatment_effect(effect_year)
    treatment_effect_list.append(treatment_effect_temp)
    count += 1

treatment_effect_list = np.array(treatment_effect_list).T[0]
treatment_effect_abs = np.max(np.abs(treatment_effect_list))
treatment_effect = np.mean(treatment_effect_list)
range_interval_list = []

for i in range(len(treatment_effect_list)):
    treatment_effect_temp = treatment_effect_list[i]
    treatment_effect_min = treatment_effect_temp - 2*treatment_effect_temp
    treatment_effect_max = treatment_effect_temp + 2*treatment_effect_temp
    range_interval_list_temp = np.linspace(treatment_effect_min, treatment_effect_max, split_num)
    range_interval_list.append(range_interval_list_temp)
    
range_interval_list = np.array(range_interval_list).T

#conformal_base_value = np.abs(np.array(treatment_effect_list).T[0]).sum() / np.sqrt(len(treatment_effect_list))

interval = mmscm_res.conformal_inference(range_interval_list)

if len(interval) == 0:
    interval = np.append(interval, treatment_effect_list)

mmscm_res = np.array([treatment_effect_list, np.min(interval, axis=0), np.max(interval, axis=0)])

mmscm_res_list.append(mmscm_res)


#---------------------------------------------------------------------------------------------------------------


mmscm_res = MMSCM(data, "DiSCo", target_unit_var, target_unit, target_outcome_var, target_year_var, target_year)
mmscm_res.train_param()
treated_unit_outcome, scm_pred = mmscm_res.predict(bias=False)

treatment_effect_list = []


count = 0

for effect_year in range(1, (mmscm_res.year[-1] - target_year)+1):
    treatment_effect_temp, target_treated_unit_outcome_temp, target_counterfactual_outcome_temp = mmscm_res.treatment_effect(effect_year)
    treatment_effect_list.append(treatment_effect_temp)
    count += 1

treatment_effect_list = np.array(treatment_effect_list).T[0]
treatment_effect_abs = np.max(np.abs(treatment_effect_list))
treatment_effect = np.mean(treatment_effect_list)
range_interval_list = []

for i in range(len(treatment_effect_list)):
    treatment_effect_temp = treatment_effect_list[i]
    treatment_effect_min = treatment_effect_temp - 2*treatment_effect_temp
    treatment_effect_max = treatment_effect_temp + 2*treatment_effect_temp
    range_interval_list_temp = np.linspace(treatment_effect_min, treatment_effect_max, split_num)
    range_interval_list.append(range_interval_list_temp)
    
range_interval_list = np.array(range_interval_list).T

#conformal_base_value = np.abs(np.array(treatment_effect_list).T[0]).sum() / np.sqrt(len(treatment_effect_list))

interval = mmscm_res.conformal_inference(range_interval_list)

if len(interval) == 0:
    interval = np.append(interval, treatment_effect_list)

mmscm_res = np.array([treatment_effect_list, np.min(interval, axis=0), np.max(interval, axis=0)])

mmscm_res_list.append(mmscm_res)

np.save("result/basque_res.npy", mmscm_res_list)

In [ ]:
###############################################################################
# SCPI Python Package
# Script for Empirical Illustration - Single Treated Unit
# Authors: Matias D. Cattaneo, Yingjie Feng, Filippo Palomba and Rocio Titiunik
###############################################################################

########################################
# Load SCPI_PKG package
import pandas
import numpy
import os
from copy import deepcopy
from warnings import filterwarnings
from scpi_pkg.scdata import scdata
from scpi_pkg.scdataMulti import scdataMulti
from scpi_pkg.scest import scest
from scpi_pkg.scpi import scpi
from scpi_pkg.scplot import scplot
from scpi_pkg.scplotMulti import scplotMulti

########################################
# Load database

filterwarnings("ignore")

##############################################################################
# SINGLE TREATED UNIT
##############################################################################

########################################
# Set options for data preparation
id_var = 'regionname'
outcome_var = 'gdpcap'
time_var = 'year'
features = None
cov_adj = None
period_pre = numpy.arange(1955, 1969)
period_post = numpy.arange(1970, 1997)
unit_tr = 'Basque Country (Pais Vasco)'
unit_co = list(set(data[id_var].to_list()))
unit_co = [cou for cou in unit_co if cou != 'Basque Country (Pais Vasco)']
constant = False
cointegrated_data = True



data_prep = scdata(df=data, id_var=id_var, time_var=time_var,
                   outcome_var=outcome_var, period_pre=period_pre,
                   period_post=period_post, unit_tr=unit_tr,
                   unit_co=unit_co, features=features, cov_adj=cov_adj,
                   cointegrated_data=cointegrated_data, constant=constant)


####################################
# SC - point estimation with simplex
est_si = scest(data_prep, w_constr={'name': "simplex"})
print(est_si)

est_si2 = scest(data_prep, w_constr={'p': 'L1', 'dir': '==', 'Q': 1, 'lb': 0})
print(est_si2)


####################################
# Set options for inference
w_constr = {'name': 'simplex', 'Q': 1}
u_missp = True
u_sigma = "HC1"
u_order = 1
u_lags = 0
e_method = "gaussian"
e_order = 1
e_lags = 0
e_alpha = 0.05
u_alpha = 0.05
sims = 200
cores = 1

numpy.random.seed(8894)
pi_si = scpi(data_prep, sims=sims, w_constr=w_constr, u_order=u_order, u_lags=u_lags,
             e_order=e_order, e_lags=e_lags, e_method=e_method, u_missp=u_missp,
             u_sigma=u_sigma, cores=cores, e_alpha=e_alpha, u_alpha=u_alpha)
print(pi_si)



mmscm_res = np.array([pi_si.Y_post["Basque Country (Pais Vasco)"].values - pi_si.Y_post_fit["A"].values, pi_si.Y_post["Basque Country (Pais Vasco)"].values - pi_si.CI_all_gaussian["Lower"].values, pi_si.Y_post["Basque Country (Pais Vasco)"].values - pi_si.CI_all_gaussian["Upper"].values])
mmscm_res_list.append(mmscm_res)
np.save("result/basque_res.npy", mmscm_res_list)

In [ ]:
#Import data
data = pd.read_csv(DATA_DIR + "smoking_data.csv")

target_unit_var = "state"
target_unit = "California"
taget_year_var = "year"
target_year = 1989

target_outcome_var = "cigsale"

In [ ]:
split_num  = 100

mmscm_res_list = []


for poly in [2, 3, 5, 10, 50, 100]:
    mmscm_res = MMSCM(data, "MMSCM", target_unit_var, target_unit, target_outcome_var, target_year_var, target_year, poly=poly)
    mmscm_res.train_param()
    treated_unit_outcome, scm_pred = mmscm_res.predict(bias=False)

    treatment_effect_list = []


    count = 0

    for effect_year in range(1, (mmscm_res.year[-1] - target_year)+1):
        treatment_effect_temp, target_treated_unit_outcome_temp, target_counterfactual_outcome_temp = mmscm_res.treatment_effect(effect_year)
        treatment_effect_list.append(treatment_effect_temp)
        count += 1

    treatment_effect_list = np.array(treatment_effect_list).T[0]
    treatment_effect_abs = np.max(np.abs(treatment_effect_list))
    treatment_effect = np.mean(treatment_effect_list)
    range_interval_list = []

    for i in range(len(treatment_effect_list)):
        treatment_effect_temp = treatment_effect_list[i]
        treatment_effect_min = treatment_effect_temp - 2*treatment_effect_temp
        treatment_effect_max = treatment_effect_temp + 2*treatment_effect_temp
        range_interval_list_temp = np.linspace(treatment_effect_min, treatment_effect_max, split_num)
        range_interval_list.append(range_interval_list_temp)

    range_interval_list = np.array(range_interval_list).T

    #conformal_base_value = np.abs(np.array(treatment_effect_list).T[0]).sum() / np.sqrt(len(treatment_effect_list))

    interval = mmscm_res.conformal_inference(range_interval_list)

    if len(interval) == 0:
        interval = np.append(interval, treatment_effect_list)

    mmscm_res = np.array([treatment_effect_list, np.min(interval, axis=0), np.max(interval, axis=0)])

    mmscm_res_list.append(mmscm_res)

#---------------------------------------------------------------------------------------------------------------


mmscm_res = MMSCM(data, "Abadie", target_unit_var, target_unit, target_outcome_var, target_year_var, target_year)
mmscm_res.train_param()
treated_unit_outcome, scm_pred = mmscm_res.predict(bias=False)

treatment_effect_list = []


count = 0

for effect_year in range(1, (mmscm_res.year[-1] - target_year)+1):
    treatment_effect_temp, target_treated_unit_outcome_temp, target_counterfactual_outcome_temp = mmscm_res.treatment_effect(effect_year)
    treatment_effect_list.append(treatment_effect_temp)
    count += 1

treatment_effect_list = np.array(treatment_effect_list).T[0]
treatment_effect_abs = np.max(np.abs(treatment_effect_list))
treatment_effect = np.mean(treatment_effect_list)
range_interval_list = []

for i in range(len(treatment_effect_list)):
    treatment_effect_temp = treatment_effect_list[i]
    treatment_effect_min = treatment_effect_temp - 2*treatment_effect_temp
    treatment_effect_max = treatment_effect_temp + 2*treatment_effect_temp
    range_interval_list_temp = np.linspace(treatment_effect_min, treatment_effect_max, split_num)
    range_interval_list.append(range_interval_list_temp)
    
range_interval_list = np.array(range_interval_list).T

#conformal_base_value = np.abs(np.array(treatment_effect_list).T[0]).sum() / np.sqrt(len(treatment_effect_list))

interval = mmscm_res.conformal_inference(range_interval_list)

if len(interval) == 0:
    interval = np.append(interval, treatment_effect_list)

mmscm_res = np.array([treatment_effect_list, np.min(interval, axis=0), np.max(interval, axis=0)])

mmscm_res_list.append(mmscm_res)


#---------------------------------------------------------------------------------------------------------------


mmscm_res = MMSCM(data, "DiSCo", target_unit_var, target_unit, target_outcome_var, target_year_var, target_year)
mmscm_res.train_param()
treated_unit_outcome, scm_pred = mmscm_res.predict(bias=False)

treatment_effect_list = []


count = 0

for effect_year in range(1, (mmscm_res.year[-1] - target_year)+1):
    treatment_effect_temp, target_treated_unit_outcome_temp, target_counterfactual_outcome_temp = mmscm_res.treatment_effect(effect_year)
    treatment_effect_list.append(treatment_effect_temp)
    count += 1

treatment_effect_list = np.array(treatment_effect_list).T[0]
treatment_effect_abs = np.max(np.abs(treatment_effect_list))
treatment_effect = np.mean(treatment_effect_list)
range_interval_list = []

for i in range(len(treatment_effect_list)):
    treatment_effect_temp = treatment_effect_list[i]
    treatment_effect_min = treatment_effect_temp - 2*treatment_effect_temp
    treatment_effect_max = treatment_effect_temp + 2*treatment_effect_temp
    range_interval_list_temp = np.linspace(treatment_effect_min, treatment_effect_max, split_num)
    range_interval_list.append(range_interval_list_temp)
    
range_interval_list = np.array(range_interval_list).T

#conformal_base_value = np.abs(np.array(treatment_effect_list).T[0]).sum() / np.sqrt(len(treatment_effect_list))

interval = mmscm_res.conformal_inference(range_interval_list)

if len(interval) == 0:
    interval = np.append(interval, treatment_effect_list)

mmscm_res = np.array([treatment_effect_list, np.min(interval, axis=0), np.max(interval, axis=0)])

mmscm_res_list.append(mmscm_res)

np.save("result/tabacco_res.npy", mmscm_res_list)

In [ ]:
###############################################################################
# SCPI Python Package
# Script for Empirical Illustration - Single Treated Unit
# Authors: Matias D. Cattaneo, Yingjie Feng, Filippo Palomba and Rocio Titiunik
###############################################################################

########################################
# Load SCPI_PKG package
import pandas
import numpy
import os
from copy import deepcopy
from warnings import filterwarnings
from scpi_pkg.scdata import scdata
from scpi_pkg.scdataMulti import scdataMulti
from scpi_pkg.scest import scest
from scpi_pkg.scpi import scpi
from scpi_pkg.scplot import scplot
from scpi_pkg.scplotMulti import scplotMulti

########################################
# Load database

filterwarnings("ignore")

##############################################################################
# SINGLE TREATED UNIT
##############################################################################

########################################
# Set options for data preparation
id_var = 'state'
outcome_var = 'cigsale'
time_var = 'year'
features = None
cov_adj = None
period_pre = numpy.arange(1970, 1988)
period_post = numpy.arange(1989, 2000)
unit_tr = 'California'
unit_co = list(set(data[id_var].to_list()))
unit_co = [cou for cou in unit_co if cou != 'California']
constant = False
cointegrated_data = True
data["year"] = np.array(data["year"], np.int64)


data_prep = scdata(df=data, id_var=id_var, time_var=time_var,
                   outcome_var=outcome_var, period_pre=period_pre,
                   period_post=period_post, unit_tr=unit_tr,
                   unit_co=unit_co, features=features, cov_adj=cov_adj,
                   cointegrated_data=cointegrated_data, constant=constant)


####################################
# SC - point estimation with simplex
est_si = scest(data_prep, w_constr={'name': "simplex"})
print(est_si)

est_si2 = scest(data_prep, w_constr={'p': 'L1', 'dir': '==', 'Q': 1, 'lb': 0})
print(est_si2)


####################################
# Set options for inference
w_constr = {'name': 'simplex', 'Q': 1}
u_missp = True
u_sigma = "HC1"
u_order = 1
u_lags = 0
e_method = "gaussian"
e_order = 1
e_lags = 0
e_alpha = 0.05
u_alpha = 0.05
sims = 200
cores = 1

numpy.random.seed(8894)
pi_si = scpi(data_prep, sims=sims, w_constr=w_constr, u_order=u_order, u_lags=u_lags,
             e_order=e_order, e_lags=e_lags, e_method=e_method, u_missp=u_missp,
             u_sigma=u_sigma, cores=cores, e_alpha=e_alpha, u_alpha=u_alpha)
print(pi_si)



mmscm_res = np.array([pi_si.Y_post["California"].values - pi_si.Y_post_fit["A"].values, pi_si.Y_post["California"].values - pi_si.CI_all_gaussian["Lower"].values, pi_si.Y_post["California"].values - pi_si.CI_all_gaussian["Upper"].values])
mmscm_res_list.append(mmscm_res)
np.save("result/tabacco_res.npy", mmscm_res_list)